# Generador 3D Multi-Vista ULTRA-HD (Hunyuan3D-2mv @ Octree 384 + Realce de Detalles) -> STL (Peana 30x30 mm)
Esta versión mejorada:
1. **Reescala y enfoca** (`1024x1024` Lanczos + UnsharpMask) las vistas frontal y trasera antes de enviarlas al modelo.
2. Ejecuta **`tencent/Hunyuan3D-2mv` al máximo de resolución del Space** (`octree_resolution=384`, `steps=50`, `guidance_scale=6.5`, `num_chunks=20000`).
3. **Proyecta el micro-relieve HD** (ojos, cejas, labios, costura del pañuelo, placas del peto y vendas) de las imágenes originales sobre la malla 3D y escala la peana a **30.0 × 30.0 mm**.

In [ ]:
!pip install -q gradio_client trimesh numpy scipy pillow
!git clone https://github.com/inigoku/mu.git /content/mu || (cd /content/mu && git pull)

In [ ]:
from pathlib import Path
import numpy as np
import trimesh
from PIL import Image, ImageFilter
from scipy import ndimage
from gradio_client import Client, handle_file
from google.colab import files

front_orig = Path('/content/mu/elektra_frontal.png')
back_orig = Path('/content/mu/elektra_trasera.png')

# 1. Super-resolución y enfoque a 1024x1024 para que DINOv2 capte el máximo detalle
def upscale_for_ai(img_path: Path, out_path: Path, size: int = 1024) -> Path:
    im = Image.open(img_path).convert('RGBA')
    up = im.resize((size, size), Image.Resampling.LANCZOS)
    rgb = up.convert('RGB').filter(ImageFilter.UnsharpMask(radius=2.0, percent=140, threshold=2))
    res = Image.merge('RGBA', (*rgb.split(), up.split()[3]))
    res.save(out_path)
    return out_path

front_1024 = upscale_for_ai(front_orig, Path('/content/elektra_frontal_1024.png'))
back_1024 = upscale_for_ai(back_orig, Path('/content/elektra_trasera_1024.png'))

print('Conectando con tencent/Hunyuan3D-2mv en modo ALTA RESOLUCIÓN (octree=384, steps=50)...')
client = Client('tencent/Hunyuan3D-2mv')
result = client.predict(
    image=handle_file(str(front_1024)),
    mv_image_front=handle_file(str(front_1024)),
    mv_image_back=handle_file(str(back_1024)),
    mv_image_left=None,
    mv_image_right=None,
    steps=50,
    guidance_scale=6.5,
    seed=1234,
    octree_resolution=384,
    check_box_rembg=False,
    num_chunks=20000,
    randomize_seed=False,
    api_name='/shape_generation'
)

mesh_path = result[0] if isinstance(result, (list, tuple)) else result
if isinstance(mesh_path, dict) and 'value' in mesh_path:
    mesh_path = mesh_path['value']

mesh = trimesh.load(mesh_path, force='mesh')
mesh.merge_vertices()
v = mesh.vertices.copy()
if mesh.extents[1] > mesh.extents[2]:
    mesh.vertices = np.column_stack([v[:, 0], -v[:, 2], v[:, 1]])

# 2. Escalar peana a 30.0 x 30.0 mm
mesh.vertices[:, 2] -= mesh.bounds[0, 2]
total_h = mesh.extents[2]
base_verts = mesh.vertices[mesh.vertices[:, 2] <= 0.12 * total_h]
base_w = float(base_verts[:, 0].max() - base_verts[:, 0].min())
base_d = float(base_verts[:, 1].max() - base_verts[:, 1].min())

sx = 30.0 / max(base_w, 1e-5)
sy = 30.0 / max(base_d, 1e-5)
sz = sx
cx = 0.5 * float(base_verts[:, 0].max() + base_verts[:, 0].min())
cy = 0.5 * float(base_verts[:, 1].max() + base_verts[:, 1].min())

mesh.vertices[:, 0] = (mesh.vertices[:, 0] - cx) * sx
mesh.vertices[:, 1] = (mesh.vertices[:, 1] - cy) * sy
mesh.vertices[:, 2] *= sz

# 3. Proyectar micro-detalle HD de las imágenes originales sobre la malla
front_rgba = np.array(Image.open(front_orig).resize((720, 660), Image.Resampling.LANCZOS), dtype=np.float32)
back_rgba = np.array(Image.open(back_orig).resize((720, 660), Image.Resampling.LANCZOS), dtype=np.float32)[:, ::-1, :]
H, W, _ = front_rgba.shape
f_mask = front_rgba[:, :, 3] > 127
b_mask = back_rgba[:, :, 3] > 127

def get_clean_crease_detail(rgba, mask):
    lum = (0.2126 * rgba[:,:,0] + 0.7152 * rgba[:,:,1] + 0.0722 * rgba[:,:,2]) / 255.0
    vals = lum[mask]
    p2, p98 = np.percentile(vals, [2, 98])
    lum = np.clip((lum - p2) / max(p98 - p2, 1e-5), 0.0, 1.0)
    lum_s = ndimage.gaussian_filter(lum, sigma=1.4)
    mid = lum_s - ndimage.gaussian_filter(lum_s, sigma=5.5)
    dist = ndimage.distance_transform_edt(mask)
    taper = np.clip((dist - 4.0) / 8.0, 0.0, 1.0)
    return ndimage.gaussian_filter(mid * taper, sigma=0.8)

f_det = get_clean_crease_detail(front_rgba, f_mask)
b_det = get_clean_crease_detail(back_rgba, b_mask)

v = mesh.vertices.copy()
vn = mesh.vertex_normals.copy()
fy_idx = np.where(f_mask)[0]
fy_min, fy_max = fy_idx.min(), fy_idx.max()
z_min, z_max = v[:, 2].min(), v[:, 2].max()
img_y = np.clip(np.round(fy_max - (v[:, 2] - z_min) / (z_max - z_min) * (fy_max - fy_min)).astype(np.int32), 0, H - 1)

f_row_x0, f_row_x1 = np.zeros(H, dtype=np.float32), np.ones(H, dtype=np.float32) * (W - 1)
b_row_x0, b_row_x1 = np.zeros(H, dtype=np.float32), np.ones(H, dtype=np.float32) * (W - 1)
for y in range(H):
    xs = np.where(f_mask[y])[0]
    if len(xs) > 2: f_row_x0[y], f_row_x1[y] = xs[0], xs[-1]
    xs_b = np.where(b_mask[y])[0]
    if len(xs_b) > 2: b_row_x0[y], b_row_x1[y] = xs_b[0], xs_b[-1]

mesh_row_x0, mesh_row_x1 = np.full(H, np.inf, dtype=np.float32), np.full(H, -np.inf, dtype=np.float32)
np.minimum.at(mesh_row_x0, img_y, v[:, 0])
np.maximum.at(mesh_row_x1, img_y, v[:, 0])
valid_r = np.isfinite(mesh_row_x0)
mesh_row_x0[valid_r] = ndimage.gaussian_filter1d(mesh_row_x0[valid_r], sigma=3.0)
mesh_row_x1[valid_r] = ndimage.gaussian_filter1d(mesh_row_x1[valid_r], sigma=3.0)

t_x = np.clip((v[:, 0] - mesh_row_x0[img_y]) / np.maximum(mesh_row_x1[img_y] - mesh_row_x0[img_y], 1e-4), 0.0, 1.0)
ux_f = np.clip(np.round(f_row_x0[img_y] + t_x * (f_row_x1[img_y] - f_row_x0[img_y])).astype(np.int32), 0, W - 1)
ux_b = np.clip(np.round(b_row_x0[img_y] + t_x * (b_row_x1[img_y] - b_row_x0[img_y])).astype(np.int32), 0, W - 1)

y_front_buf, y_back_buf = np.full((H, W), np.inf, dtype=np.float32), np.full((H, W), -np.inf, dtype=np.float32)
np.minimum.at(y_front_buf, (img_y, ux_f), v[:, 1])
np.maximum.at(y_back_buf, (img_y, ux_b), v[:, 1])
y_front_buf = ndimage.minimum_filter(y_front_buf, size=3)
y_back_buf = ndimage.maximum_filter(y_back_buf, size=3)

vis_front = np.clip(1.0 - np.abs(v[:, 1] - y_front_buf[img_y, ux_f]) / 1.8, 0.0, 1.0) * np.clip(-vn[:, 1] - 0.15, 0.0, 1.0)
vis_back = np.clip(1.0 - np.abs(v[:, 1] - y_back_buf[img_y, ux_b]) / 1.8, 0.0, 1.0) * np.clip(vn[:, 1] - 0.15, 0.0, 1.0)

total_disp = (vis_front * f_det[img_y, ux_f] + vis_back * b_det[img_y, ux_b]) * 0.75
mesh.vertices = v + vn * total_disp[:, None]
trimesh.smoothing.filter_laplacian(mesh, lamb=0.20, iterations=2)

output_stl = '/content/elektra_hunyuan3d_ULTRA_HD_30x30.stl'
mesh.export(output_stl)
print(f'STL ULTRA-HD generado: {output_stl} | Triángulos: {len(mesh.faces):,} | Dimensiones: {mesh.extents.round(1)} mm')
files.download(output_stl)